# Car Price Prediction Using Linear, Ridge, and Lasso Regression

## Introduction

The purpose of this assignment is to build and compare three regression models that predict the price of a car. The models used are Linear Regression, Ridge Regression, and Lasso Regression. All three models are trained and tested using the same data split so that their performance can be compared fairly.

The target variable in this dataset is `price`. The models are evaluated using Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R² score.

**Dataset source:** Car Price Prediction dataset from Kaggle  
**Dataset file:** `CarPrice_Assignment.csv`


## 1. Import the Required Libraries

In [1]:
# Libraries used for data handling
import os
import numpy as np
import pandas as pd

# Scikit-learn tools used for preprocessing, model training, and evaluation
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Display settings make the output easier to read
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")


## 2. Load the Dataset

The code below looks for the CSV file in the same folder as the notebook. When the notebook is opened in Google Colab, it also provides an option to upload the file directly.


In [2]:
# Common file names are checked so the notebook works even if the file was renamed.
possible_files = [
    "CarPrice_Assignment.csv",
    "car_price_prediction.csv",
    "CarPrice.csv"
]

file_path = next(
    (file_name for file_name in possible_files if os.path.exists(file_name)),
    None
)

# If the file is not available locally, request an upload in Google Colab.
if file_path is None:
    try:
        from google.colab import files

        print("Please upload the CarPrice_Assignment.csv file.")
        uploaded = files.upload()
        file_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "The dataset was not found. Place CarPrice_Assignment.csv "
            "in the same folder as this notebook and run the cell again."
        )

car_data = pd.read_csv(file_path)

print(f"Dataset loaded successfully from: {file_path}")
print(f"The dataset contains {car_data.shape[0]} rows and {car_data.shape[1]} columns.")

car_data.head()


Dataset loaded successfully from: CarPrice_Assignment.csv
The dataset contains 205 rows and 26 columns.


,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6000,168.8000,64.1000,48.8000,2548,dohc,four,130,mpfi,3.4700,2.6800,9.0000,111,5000,21,27,13495.0000
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6000,168.8000,64.1000,48.8000,2548,dohc,four,130,mpfi,3.4700,2.6800,9.0000,111,5000,21,27,16500.0000
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5000,171.2000,65.5000,52.4000,2823,ohcv,six,152,mpfi,2.6800,3.4700,9.0000,154,5000,19,26,16500.0000
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8000,176.6000,66.2000,54.3000,2337,ohc,four,109,mpfi,3.1900,3.4000,10.0000,102,5500,24,30,13950.0000
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4000,176.6000,66.4000,54.3000,2824,ohc,five,136,mpfi,3.1900,3.4000,8.0000,115,5500,18,22,17450.0000


## 3. Prepare the Data

Duplicate rows are removed before modeling. The `price` column is separated as the target variable, while the remaining useful columns are used as input features.

`car_ID` is removed because it is only a unique identifier. `CarName` is also removed because it contains detailed model names that can create too many categories and may cause the model to memorize individual cars rather than learn general pricing patterns.


In [3]:
# Remove duplicate records, if any.
car_data = car_data.drop_duplicates().copy()

# Make sure the required target column is available.
if "price" not in car_data.columns:
    raise KeyError("The dataset must contain a column named 'price'.")

# Remove identifier-style columns that are not useful for general prediction.
columns_to_remove = [
    column for column in ["car_ID", "CarName"]
    if column in car_data.columns
]

# X contains the predictor variables, and y contains the car price.
X = car_data.drop(columns=["price"] + columns_to_remove)
y = car_data["price"]

print("Removed columns:", columns_to_remove)
print("Number of predictor columns:", X.shape[1])
print("Number of observations:", X.shape[0])


Removed columns: ['car_ID', 'CarName']
Number of predictor columns: 23
Number of observations: 205


## 4. Create the Train/Test Split

The dataset is divided into training and testing sets. Eighty percent of the data is used to train the models, while the remaining twenty percent is used to evaluate how well the models perform on unseen data.

A fixed `random_state` is used so that the same split and results can be reproduced each time the notebook is run.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training observations: {X_train.shape[0]}")
print(f"Testing observations: {X_test.shape[0]}")


Training observations: 164
Testing observations: 41


## 5. Preprocess Numerical and Categorical Features

The dataset contains both numerical and categorical columns. Missing numerical values are filled with the median, and missing categorical values are filled with the most frequent category.

Numerical variables are standardized because Ridge and Lasso regression apply penalties to model coefficients. Standardization places the numerical features on a similar scale, which allows regularization to work more fairly. Categorical variables are converted into numerical form using one-hot encoding.


In [5]:
# Separate numerical and categorical column names.
numerical_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X.select_dtypes(exclude=["number"]).columns.tolist()

# Numerical preprocessing:
# 1. Replace missing values with the median.
# 2. Standardize the variables.
numerical_preprocessing = Pipeline(
    steps=[
        ("missing_value_handler", SimpleImputer(strategy="median")),
        ("standardization", StandardScaler())
    ]
)

# Categorical preprocessing:
# 1. Replace missing values with the most common category.
# 2. Convert text categories into dummy variables.
categorical_preprocessing = Pipeline(
    steps=[
        ("missing_value_handler", SimpleImputer(strategy="most_frequent")),
        (
            "one_hot_encoding",
            OneHotEncoder(handle_unknown="ignore", drop="first")
        )
    ]
)

# Apply the correct preprocessing steps to each type of column.
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_preprocessing, numerical_columns),
        ("categorical", categorical_preprocessing, categorical_columns)
    ]
)

print(f"Numerical columns: {len(numerical_columns)}")
print(f"Categorical columns: {len(categorical_columns)}")


Numerical columns: 14
Categorical columns: 9


## 6. Train the Three Regression Models

Three models are trained using the same training data:

- **Linear Regression** is the basic regression model without regularization.
- **Ridge Regression** adds an L2 penalty that reduces large coefficients but usually keeps all features.
- **Lasso Regression** adds an L1 penalty that can reduce some coefficients to zero and therefore perform feature selection.

The same preprocessing pipeline is used for every model to keep the comparison consistent.


In [6]:
# Define the three models required for the assignment.
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01, max_iter=100000)
}

model_results = []
trained_models = {}

for model_name, regression_model in models.items():

    # Combine preprocessing and model training into one pipeline.
    model_pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            ("regression_model", regression_model)
        ]
    )

    # Train the model using the training data.
    model_pipeline.fit(X_train, y_train)

    # Predict car prices for the unseen test data.
    predicted_prices = model_pipeline.predict(X_test)

    # Calculate the four required performance measures.
    mae = mean_absolute_error(y_test, predicted_prices)
    mse = mean_squared_error(y_test, predicted_prices)
    rmse = np.sqrt(mse)
    r_squared = r2_score(y_test, predicted_prices)

    model_results.append({
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R²": r_squared
    })

    trained_models[model_name] = model_pipeline

print("Linear, Ridge, and Lasso regression models were trained successfully.")


Linear, Ridge, and Lasso regression models were trained successfully.


/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## 7. Compare Model Performance

The table below compares the three models using all four required evaluation metrics.

- Lower MAE, MSE, and RMSE values indicate smaller prediction errors.
- A higher R² value indicates that the model explains more of the variation in car prices.


In [7]:
comparison_table = pd.DataFrame(model_results)

# Arrange the models from highest to lowest R² score.
comparison_table = comparison_table.sort_values(
    by="R²",
    ascending=False
).reset_index(drop=True)

comparison_table


,Model,MAE,MSE,RMSE,R²
0,Linear Regression,2089.3827,8482008.4844,2912.3888,0.8926
1,Lasso Regression,2089.4605,8483700.7968,2912.6793,0.8925
2,Ridge Regression,2066.4677,9802517.0469,3130.8972,0.8758


## 8. Conclusion

The following cell creates a short conclusion directly from the results. This prevents the written conclusion from naming the wrong model when the notebook is run on a different computer or with a slightly different version of the dataset.


In [8]:
# Identify the model with the highest R² score.
best_result = comparison_table.iloc[0]

best_model_name = best_result["Model"]
best_mae = best_result["MAE"]
best_rmse = best_result["RMSE"]
best_r_squared = best_result["R²"]

# Write a natural conclusion based on the winning model.
if best_model_name == "Linear Regression":
    conclusion = (
        f"Linear Regression produced the best overall test performance, with an "
        f"R² score of {best_r_squared:.4f}, an MAE of {best_mae:.2f}, and an "
        f"RMSE of {best_rmse:.2f}. In this dataset, Ridge and Lasso regularization "
        f"did not improve the results, although they still reduced the influence "
        f"of large coefficients and may help control overfitting."
    )

elif best_model_name == "Ridge Regression":
    conclusion = (
        f"Ridge Regression was the best-performing model, with an R² score of "
        f"{best_r_squared:.4f}, an MAE of {best_mae:.2f}, and an RMSE of "
        f"{best_rmse:.2f}. The Ridge penalty improved the model's ability to "
        f"generalize by shrinking large coefficients while keeping all of the "
        f"predictor variables in the model."
    )

else:
    conclusion = (
        f"Lasso Regression achieved the best test performance, with an R² score "
        f"of {best_r_squared:.4f}, an MAE of {best_mae:.2f}, and an RMSE of "
        f"{best_rmse:.2f}. Lasso regularization helped by shrinking less useful "
        f"coefficients and could reduce some of them to zero, which also acts as "
        f"a form of feature selection."
    )

print(conclusion)


Linear Regression produced the best overall test performance, with an R² score of 0.8926, an MAE of 2089.38, and an RMSE of 2912.39. In this dataset, Ridge and Lasso regularization did not improve the results, although they still reduced the influence of large coefficients and may help control overfitting.


## Final Summary

This assignment used an 80/20 train/test split and compared Linear Regression, Ridge Regression, and Lasso Regression. Each model was evaluated using MAE, MSE, RMSE, and R², and the best model was selected using its overall performance on the test data.
